In [3]:
!pip install openmeteo-requests

   ---------------------------------------- 0.0/707.8 kB ? eta -:--:--
   ---------------------------------------- 707.8/707.8 kB 8.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 9.5 MB/s  0:00:00

   ---------------------------------------- 0/8 [flatbuffers]
   ---------------------------------------- 0/8 [flatbuffers]
   ----- ---------------------------------- 1/8 [wassima]
   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   --------------- ------------------------ 3/8 [openmeteo-sdk]
   --------------- ------------------------ 3/8 [openmeteo-sdk]
   -------------------- ------------------- 4/8 [jh2]
   -------------------- ------------------- 4/8 [jh2]
   ------------------

In [1]:
from datetime import datetime
import openmeteo_requests

In [2]:
class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.max_speed:
            raise StopIteration
        self.current = min(self.current + self.step, self.max_speed)
        return self.current


class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed=0, step=10):
        self.current = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= self.min_speed:
            raise StopIteration
        self.current = max(self.current - self.step, self.min_speed)
        return self.current

In [3]:
class Car:

    cars_on_road = 0  

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed


        if current_speed > 0:
            self.state = True
            Car.cars_on_road += 1
        else:
            self.state = False

   
    
    def accelerate(self, upper_border=None, step=10):

        if not self.state:
            self.state = True
            Car.cars_on_road += 1

        start_speed = self.current_speed

        if upper_border is not None and upper_border <= self.max_speed:

            iterator = IncreaseSpeed(self.current_speed, upper_border, step)

            for speed in iterator:
                print("INFO: Speed increases by 10")
                self.current_speed = speed

        else:
            self.current_speed = min(self.current_speed + step, self.max_speed)

        print(f"Speed: {start_speed} -> {self.current_speed}")

   
    
    def brake(self, lower_border=None, step=10):

        start_speed = self.current_speed

        if lower_border is not None and lower_border >= 0:

            iterator = DecreaseSpeed(self.current_speed, lower_border, step)

            for speed in iterator:
                print("INFO: Speed decreases by 10")
                self.current_speed = speed

        else:
            self.current_speed = max(self.current_speed - step, 0)

        print(f"Speed: {start_speed} -> {self.current_speed}")


    
    def parking(self):

        if not self.state:
            return

        start_speed = self.current_speed
        self.current_speed = 0

        print(f"Speed: {start_speed} -> 0")
        print("Car parked")

        self.state = False
        Car.cars_on_road -= 1
        
        
    @classmethod
    def total_cars(cls):
        return cls.cars_on_road

    @staticmethod
    def show_weather():

        openmeteo = openmeteo_requests.Client()

        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }

        response = openmeteo.weather_api(url, params=params)[0]

        current = response.Current()

        temp = current.Variables(0).Value()
        apparent = current.Variables(1).Value()
        rain = current.Variables(2).Value()
        wind = current.Variables(3).Value()

        print(f"Current temperature: {round(temp, 0)} C")
        print(f"Current apparent_temperature: {round(apparent, 0)} C")
        print(f"Current rain: {rain} mm")
        print(f"Current wind_speed: {round(wind, 1)} m/s")

In [4]:
if __name__ == "__main__":

    car1 = Car(100, 20)
    car2 = Car(60, 30)
    car3 = Car(100, 0)

    print("Total cars on road:", Car.total_cars())

    car1.accelerate(100)
    car2.accelerate(50)

    print("Speed of car1:", car1.current_speed)
    print("Speed of car2:", car2.current_speed)

    car1.brake(10)
    car2.brake(0)

    print("Total cars on road:", Car.total_cars())

    car2.parking()

    print("Total cars on road:", Car.total_cars())

    car3.accelerate(80)
    car3.show_weather()

    print("Total cars on road:", Car.total_cars())

    car2.accelerate(10)

    print("Total cars on road:", Car.total_cars())

    Car.show_weather()

Total cars on road: 2
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
Speed: 20 -> 100
INFO: Speed increases by 10
INFO: Speed increases by 10
Speed: 30 -> 50
Speed of car1: 100
Speed of car2: 50
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
Speed: 100 -> 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
Speed: 50 -> 0
Total cars on road: 2
Speed: 0 -> 0
Car parked
Total cars on road: 1
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases b